In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import random

random.seed(42)
np.random.seed(42)

# --- Resolve project paths ---
def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'stimuli' / 'words' / 'death_words.csv').exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find the project root starting from {start_path}. "
        "Expected to find stimuli/words/death_words.csv in the workspace tree."
    )

PROJECT_DIR = find_project_root(Path.cwd())
WORDS_DIR = PROJECT_DIR / 'stimuli' / 'words'
# FACES_DIR = PROJECT_DIR / 'stimuli' / 'faces'
FACES_DIR = PROJECT_DIR / 'stimuli' / 'faces' / 'GenAI_faces'

print(f'Using project directory: {PROJECT_DIR}')

# --- Parameters ---
N_BLOCKS = 3
TRIALS_PER_BLOCK = 120
TARGET_TRIALS_PER_BLOCK = 30
NON_TARGET_PER_BLOCK = TRIALS_PER_BLOCK - TARGET_TRIALS_PER_BLOCK  # 90
CONDITIONS = ['DS', 'DO', 'NS', 'NO']  # death/self, death/other, neg/self, neg/other

PRIME_MAP  = {'DS': 'DEATH', 'DO': 'DEATH', 'NS': 'NEGATIVE', 'NO': 'NEGATIVE'}
IDENTITY_MAP = {'DS': 'SELF', 'DO': 'OTHER', 'NS': 'SELF', 'NO': 'OTHER'}

STD_IMAGE  = {'SELF': str(FACES_DIR / 'self.png'),  'OTHER': str(FACES_DIR / 'other.png')}
TGT_IMAGE  = {'SELF': str(FACES_DIR / 'self_target.png'), 'OTHER': str(FACES_DIR / 'other_target.png')}
DEV_IMAGE  = str(FACES_DIR / 'morph50.png')
DEV_TGT_IMAGE = str(FACES_DIR / 'morph50_target.png')

# Load words
death_words = pd.read_csv(WORDS_DIR / 'death_words.csv')['word'].tolist()
negative_words = pd.read_csv(WORDS_DIR / 'negative_words.csv')['word'].tolist()

# --- Word cycling (each word appears roughly equally) ---
def make_word_cycle(words, n_needed):
    """Cycle through word list, shuffling each pass, to fill n_needed slots."""
    result = []
    while len(result) < n_needed:
        shuffled = words[:]
        random.shuffle(shuffled)
        result.extend(shuffled)
    return result[:n_needed]

# Count how many death/negative trials we need across all blocks
# Non-target: 67.5 per condition = 270 non-target total, 135 death, 135 negative
# Target: 90 total, distributed across conditions (paper doesn't specify, assume equal: ~22-23 per cond)
# Total death trials: ~90 + ~45 targets = ~180; same for negative
# We'll assign words after building the trial list, so count first

rows = []

for block in range(1, N_BLOCKS + 1):
    # Build 90 non-target trials: ~22-23 per condition (balance as evenly as possible)
    non_target_conds = []
    for i in range(NON_TARGET_PER_BLOCK):
        non_target_conds.append(CONDITIONS[i % 4])
    random.shuffle(non_target_conds)

    # Build 30 target trials: ~7-8 per condition
    target_conds = []
    for i in range(TARGET_TRIALS_PER_BLOCK):
        target_conds.append(CONDITIONS[i % 4])
    random.shuffle(target_conds)

    # Combine and shuffle
    block_trials = (
        [(c, False) for c in non_target_conds] +
        [(c, True) for c in target_conds]
    )
    random.shuffle(block_trials)

    for trial_idx, (cond, is_target) in enumerate(block_trials):
        identity = IDENTITY_MAP[cond]
        n_std = random.randint(3, 6)  # uniform 3-6

        if is_target:
            # Target can appear at any position (1..n_std for standard slot, n_std+1 for deviant)
            target_pos = random.randint(1, n_std + 1)
        else:
            target_pos = -1

        rows.append({
            'block':              block,
            'trial_in_block':     trial_idx + 1,
            'condCode':           cond,
            'primeType':          PRIME_MAP[cond],
            'identity':           identity,
            'word':               '',          # filled below
            'nStandards':         n_std,
            'isTarget':           int(is_target),
            'targetPosition':     target_pos,
            'standardImage':      STD_IMAGE[identity],
            'deviantImage':       DEV_IMAGE,
            'standardTargetImage': TGT_IMAGE[identity],
            'deviantTargetImage': DEV_TGT_IMAGE,
        })

df = pd.DataFrame(rows)

# --- Assign words ---
death_trials_idx = df[df['primeType'] == 'DEATH'].index.tolist()
negative_trials_idx = df[df['primeType'] == 'NEGATIVE'].index.tolist()

death_cycle = make_word_cycle(death_words, len(death_trials_idx))
negative_cycle = make_word_cycle(negative_words, len(negative_trials_idx))

for i, idx in enumerate(death_trials_idx):
    df.at[idx, 'word'] = death_cycle[i]
for i, idx in enumerate(negative_trials_idx):
    df.at[idx, 'word'] = negative_cycle[i]

# --- Verification ---
total = len(df)
n_targets = df['isTarget'].sum()
print(f"Total trials: {total}  (expected 360)")
print(f"Target trials: {n_targets}  (expected 90)")
print(f"\nCondition counts (all trials):\n{df['condCode'].value_counts().sort_index()}")
print(f"\nNon-target condition counts:\n{df[df['isTarget']==0]['condCode'].value_counts().sort_index()}")

# Verify 82/18 standard/deviant ratio
total_face_events = (df['nStandards'] + 1).sum()
total_deviants = len(df)
total_standards = (df['nStandards']).sum()
print(f"\nStandard images: {total_standards} ({100*total_standards/total_face_events:.1f}%)")
print(f"Deviant images:  {total_deviants} ({100*total_deviants/total_face_events:.1f}%)")
print(f"(Paper reports 82% / 18%)")

# --- Save ---
output_path = PROJECT_DIR / 'conditions' / 'main_trials.csv'
df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\nSaved to {output_path}")

Using project directory: c:\Users\AvivaLab\code\EEG_experiments\vMMR_EEG_PsychoPy_experiment
Total trials: 360  (expected 360)
Target trials: 90  (expected 90)

Condition counts (all trials):
condCode
DO    93
DS    93
NO    87
NS    87
Name: count, dtype: int64

Non-target condition counts:
condCode
DO    69
DS    69
NO    66
NS    66
Name: count, dtype: int64

Standard images: 1643 (82.0%)
Deviant images:  360 (18.0%)
(Paper reports 82% / 18%)

Saved to c:\Users\AvivaLab\code\EEG_experiments\vMMR_EEG_PsychoPy_experiment\conditions\main_trials.csv


In [2]:
from pathlib import Path
import pandas as pd
import random

random.seed(99)

# --- Resolve project paths ---
def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'stimuli' / 'words' / 'death_words.csv').exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find the project root starting from {start_path}. "
        "Expected to find stimuli/words/death_words.csv in the workspace tree."
    )

PROJECT_DIR = find_project_root(Path.cwd())
WORDS_DIR = PROJECT_DIR / 'stimuli' / 'words'
# FACES_DIR = PROJECT_DIR / 'stimuli' / 'faces'
FACES_DIR = PROJECT_DIR / 'stimuli' / 'faces' / 'GenAI_faces'

print(f'Using project directory: {PROJECT_DIR}')

CONDITIONS = ['DS', 'DO', 'NS', 'NO']
PRIME_MAP    = {'DS': 'DEATH', 'DO': 'DEATH', 'NS': 'NEGATIVE', 'NO': 'NEGATIVE'}
IDENTITY_MAP = {'DS': 'SELF',  'DO': 'OTHER', 'NS': 'SELF',     'NO': 'OTHER'}
STD_IMAGE    = {'SELF': str(FACES_DIR / 'self.png'), 'OTHER': str(FACES_DIR / 'other.png')}
TGT_IMAGE    = {'SELF': str(FACES_DIR / 'self_target.png'), 'OTHER': str(FACES_DIR / 'other_target.png')}
DEV_IMAGE        = str(FACES_DIR / 'morph50.png')
DEV_TGT_IMAGE    = str(FACES_DIR / 'morph50_target.png')

death_words = pd.read_csv(WORDS_DIR / 'death_words.csv')['word'].tolist()
negative_words = pd.read_csv(WORDS_DIR / 'negative_words.csv')['word'].tolist()

# 16 trials: each condition x4, 4 of them targets (one per condition)
conds = CONDITIONS * 4
random.shuffle(conds)
target_conds = set(random.sample(CONDITIONS, 4))  # one target per condition
target_assigned = {c: False for c in CONDITIONS}

rows = []
d_idx, n_idx = 0, 0

for i, cond in enumerate(conds):
    identity = IDENTITY_MAP[cond]
    n_std = random.randint(3, 6)
    
    is_target = (not target_assigned[cond]) and (cond in target_conds)
    if is_target:
        target_assigned[cond] = True
        target_pos = random.randint(1, n_std + 1)
    else:
        target_pos = -1

    if PRIME_MAP[cond] == 'DEATH':
        word = death_words[d_idx % len(death_words)]
        d_idx += 1
    else:
        word = negative_words[n_idx % len(negative_words)]
        n_idx += 1

    rows.append({
        'block': 0,
        'trial_in_block': i + 1,
        'condCode': cond,
        'primeType': PRIME_MAP[cond],
        'identity': identity,
        'word': word,
        'nStandards': n_std,
        'isTarget': int(is_target),
        'targetPosition': target_pos,
        'standardImage': STD_IMAGE[identity],
        'deviantImage': DEV_IMAGE,
        'standardTargetImage': TGT_IMAGE[identity],
        'deviantTargetImage': DEV_TGT_IMAGE,
    })


df_prac = pd.DataFrame(rows)
print(df_prac[['condCode','primeType','identity','word','nStandards','isTarget','targetPosition']])

output_path = PROJECT_DIR / 'conditions' / 'practice_trials.csv'
df_prac.to_csv(output_path, index=False, encoding='utf-8')
print(f"\nSaved: {output_path}")

Using project directory: c:\Users\AvivaLab\code\EEG_experiments\vMMR_EEG_PsychoPy_experiment
   condCode primeType identity              word  nStandards  isTarget  \
0        DS     DEATH     SELF             אֵבֶל           6         1   
1        NO  NEGATIVE    OTHER            אֲסִיר           4         1   
2        NS  NEGATIVE     SELF         בְּחִילָה           6         1   
3        DS     DEATH     SELF          אוֹבְדָן           4         0   
4        NO  NEGATIVE    OTHER          בִּיזוּי           5         0   
5        DO     DEATH    OTHER        אַזְכָּרָה           3         1   
6        DS     DEATH     SELF           אֵיידְס           6         0   
7        DO     DEATH    OTHER             אֵפֶר           5         0   
8        NO  NEGATIVE    OTHER            גֶּזֶל           6         0   
9        NS  NEGATIVE     SELF  גִילוּי עֲרָיוֹת           3         0   
10       DO     DEATH    OTHER          גִ'יהַאד           5         0   
11       NS  NEGATI